In [1]:
import pandas as pd
import httpx
from io import StringIO

In [2]:
bond_cal = (
    pd.read_excel(
        "https://www.gov.pl/attachment/f1cde310-8ed5-4a8e-b8e8-c91827a1be2f",
        header=[0, 1],
        sheet_name="ObligacjeStałoprocentowe",
    )
    .rename(
        columns={
            "Unnamed: 0_level_0": "Info",
            "Unnamed: 1_level_0": "Info",
            "Unnamed: 2_level_0": "Info",
            "Unnamed: 3_level_0": "Info",
        }
    )
    .replace({"-": pd.NA})
)
bond_cal.head(2)

Info                                        Kupon Nr 1                \
    Seria      Kod ISIN Data wykupu   Kupon Początek okresu Koniec okresu   
0  RS0799  PL0000101101  1999-07-01  0.2071      1998-07-01    1999-07-01   
1  RS1099  PL0000101119  1999-10-01  0.1800      1998-10-01    1999-10-01   

                                                            Kupon Nr 2  ...  \
  Dzień ustalenia praw Data wymagalności Odsetki (PLN) Początek okresu  ...   
0           1999-06-18        1999-07-01         20.71             NaT  ...   
1           1999-09-20        1999-10-01         18.00             NaT  ...   

      Kupon Nr 30                                                       \
  Początek okresu Koniec okresu Dzień ustalenia praw Data wymagalności   
0             NaN           NaN                  NaN               NaN   
1             NaN           NaN                  NaN               NaN   

                    Kupon Nr 31                                     \
  Odsetki (PLN) Początek okresu Koniec okresu Dzień ustalenia praw   
0           NaN             NaN           NaN                  NaN   
1           NaN             NaN           NaN                  NaN   

                                   
  Data wymagalności Odsetki (PLN)  
0               NaN           NaN  
1               NaN           NaN  

[2 rows x 159 columns]

In [3]:
info = bond_cal.loc[:, ["Info"]].stack(level=0, future_stack=True).reset_index(1, drop=True)
calendar = (
    bond_cal.loc[:, [f"Kupon Nr {num}" for num in range(1, 32)]]
    .stack(level=0, future_stack=True)
    .reset_index(1)
)
df = info.join(calendar).dropna(how="any").rename(columns={"level_1": "Numer okresu"})
df["Numer okresu"] = df["Numer okresu"].str[9:]
df.astype({
    "Początek okresu": "datetime64[ns]",
    "Koniec okresu": "datetime64[ns]",
    "Dzień ustalenia praw": "datetime64[ns]",
    "Data wymagalności": "datetime64[ns]",
    "Odsetki (PLN)": "float64",
    "Numer okresu": "int16",
    "Kod ISIN": "string",
    "Seria": "string",
})
df.to_parquet('kalendarz_odsetkowy.parquet', index=False)

In [4]:
df.loc[df.Seria == 'DS1035']

,Seria,Kod ISIN,Data wykupu,Kupon,Numer okresu,Początek okresu,Koniec okresu,Dzień ustalenia praw,Data wymagalności,Odsetki (PLN)
118,DS1035,PL0000118188,2035-10-25,0.05,1,2024-10-25 00:00:00,2025-10-25 00:00:00,2025-10-23 00:00:00,2025-10-27 00:00:00,50.0
118,DS1035,PL0000118188,2035-10-25,0.05,2,2025-10-25 00:00:00,2026-10-25 00:00:00,2026-10-22 00:00:00,2026-10-26 00:00:00,50.0
118,DS1035,PL0000118188,2035-10-25,0.05,3,2026-10-25 00:00:00,2027-10-25 00:00:00,2027-10-21 00:00:00,2027-10-25 00:00:00,50.0
118,DS1035,PL0000118188,2035-10-25,0.05,4,2027-10-25 00:00:00,2028-10-25 00:00:00,2028-10-23 00:00:00,2028-10-25 00:00:00,50
118,DS1035,PL0000118188,2035-10-25,0.05,5,2028-10-25 00:00:00,2029-10-25 00:00:00,2029-10-23 00:00:00,2029-10-25 00:00:00,50
118,DS1035,PL0000118188,2035-10-25,0.05,6,2029-10-25 00:00:00,2030-10-25 00:00:00,2030-10-23 00:00:00,2030-10-25 00:00:00,50
118,DS1035,PL0000118188,2035-10-25,0.05,7,2030-10-25 00:00:00,2031-10-25 00:00:00,2031-10-23 00:00:00,2031-10-27 00:00:00,50
118,DS1035,PL0000118188,2035-10-25,0.05,8,2031-10-25 00:00:00,2032-10-25 00:00:00,2032-10-21 00:00:00,2032-10-25 00:00:00,50
118,DS1035,PL0000118188,2035-10-25,0.05,9,2032-10-25 00:00:00,2033-10-25 00:00:00,2033-10-21 00:00:00,2033-10-25 00:00:00,50
118,DS1035,PL0000118188,2035-10-25,0.05,10,2033-10-25 00:00:00,2034-10-25 00:00:00,2034-10-23 00:00:00,2034-10-25 00:00:00,50
